# Test: build_feature_store

This notebook checks that `build_feature_store` works end to end against real data, for one brand and one cutoff date.

In [1]:
import sys
sys.path.insert(0, "../src/features")
from build_features import build_feature_store, FEATURES, ID_COLUMNS, FEATURE_COLUMNS
import pandas as pd


In [2]:
df_features = build_feature_store(cutoff_date="2026-07-25", brand_id=64)
print("Shape:", df_features.shape)
print("Columns:", df_features.columns.tolist())


Shape: (14759, 34)
Columns: ['cutoff_date', 'operator', 'brandId', 'partyId', 'label_available_60d', 'days_since_last_active', 'days_since_last_deposit', 'days_since_last_bonus', 'n_active_days_last_7d', 'n_active_days_last_30d', 'n_deposit_last_7d', 'n_deposit_last_30d', 'n_bonus_last_7d', 'stake_last_7d', 'stake_last_30d', 'win_last_7d', 'win_last_30d', 'net_loss_last_7d', 'net_loss_last_30d', 'deposit_amount_last_7d', 'deposit_amount_last_30d', 'bonus_amount_last_7d', 'freq_drop_7_vs_prior7', 'deposit_drop_7_vs_prior7', 'bonus_drop_7_vs_prior7', 'stake_trend_7', 'win_trend_7', 'net_loss_trend_7', 'deposit_amount_trend_7', 'bonus_amount_trend_7', 'already_dormant_7', 'tenure_days', 'max_prior_gap_days', 'rtp_last_7d']


It took 30 minutes to compile, so I decided to save the table so I could continue working.

In [6]:
from pathlib import Path

OUTPUT_PATH = Path("../data/03_output/features_brand64_2026-07-25.parquet")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_features.to_parquet(OUTPUT_PATH)
print(f"saved {OUTPUT_PATH} ({len(df_features):,} rows, {df_features.shape[1]} columns)")


saved ../data/03_output/features_brand64_2026-07-25.parquet (14,759 rows, 34 columns)


## Check: unexpected columns, nulls and duplicate partyId

In [7]:
expected_columns = ID_COLUMNS + FEATURES

assert df_features.columns.tolist() == expected_columns, "Unexpected columns or column order"
assert not df_features.isna().any().any(), "Dataframe contains null values"
assert not df_features["partyId"].duplicated().any(), "Duplicate partyId values found"
print("OK")


OK


## Check: repeatability

A second build gives the same snapshot (row order can differ: DuckDB `DISTINCT` does not keep it).

In [ ]:
df_again = build_feature_store(cutoff_date="2026-07-25", brand_id=64)
by_player = lambda d: d.sort_values("partyId").reset_index(drop=True)
pd.testing.assert_frame_equal(by_player(df_features), by_player(df_again), check_exact=True, obj="first vs second build")
print("Repeatability: OK")

In [8]:
df_features[["tenure_days", "net_loss_last_7d", "rtp_last_7d"]].describe()


,tenure_days,net_loss_last_7d,rtp_last_7d
count,14759.000000,14759.000000,14759.000000
mean,3.814394,0.941449,0.797935
std,1.235836,2.384505,0.213126
min,0.000000,-9.557002,0.000000
25%,3.610918,0.000000,0.810572
50%,3.610918,0.000000,0.810572
75%,3.610918,2.333599,0.810572
max,7.724005,9.663322,6.706081


In [9]:
df_features


,cutoff_date,operator,brandId,partyId,label_available_60d,days_since_last_active,days_since_last_deposit,days_since_last_bonus,n_active_days_last_7d,n_active_days_last_30d,...,bonus_drop_7_vs_prior7,stake_trend_7,win_trend_7,net_loss_trend_7,deposit_amount_trend_7,bonus_amount_trend_7,already_dormant_7,tenure_days,max_prior_gap_days,rtp_last_7d
0,2026-07-25,primus,64,3378990,True,0.693147,0.693147,3.663562,1.386294,1.386294,...,0.000000,9.076847,8.877211,7.368059,7.625741,0.000000,0,3.610918,0.693147,0.819008
1,2026-07-25,primus,64,3388304,True,0.000000,3.663562,3.663562,0.693147,1.609438,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,3.610918,2.708050,0.810572
2,2026-07-25,primus,64,3398055,True,2.484907,3.663562,3.663562,0.000000,0.693147,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,3.610918,0.000000,0.810572
3,2026-07-25,primus,64,3473849,True,1.609438,3.663562,3.663562,0.693147,1.098612,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,3.610918,1.609438,0.810572
4,2026-07-25,primus,64,3523811,True,3.526361,3.555348,3.526361,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1,7.556951,0.693147,0.810572
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14754,2026-07-25,primus,64,19064831,True,1.609438,1.609438,3.663562,0.693147,0.693147,...,0.000000,2.588848,1.387316,2.333148,2.333599,0.000000,0,1.609438,0.000000,0.243949
14755,2026-07-25,primus,64,19069700,True,1.098612,1.098612,1.098612,1.098612,1.098612,...,1.098612,7.602989,7.842887,-6.299632,6.145794,6.240927,0,1.386294,0.693147,1.271255
14756,2026-07-25,primus,64,19076519,True,0.000000,0.000000,3.663562,1.098612,1.098612,...,0.000000,4.165515,3.974992,2.499478,2.499631,0.000000,0,0.693147,0.693147,0.823792
14757,2026-07-25,primus,64,19078754,True,0.000000,0.000000,0.693147,1.098612,1.098612,...,0.693147,5.631651,5.650976,-1.863534,2.985564,2.537163,0,0.693147,0.693147,1.019583
